# Creating CSV for create_datasets.py

Input CSV columns:

  accession_id  – id of the viral genome

  record_id     – id of the record in the corresponding accession

  cluster_id    – a record_id representative id

  prf_position  – position of PRF event (1-based); can be comma-separated for multiple sites

  strand        – +/- strand

  sequence      – raw DNA/RNA sequence

  split         – train / val / test

In [ ]:
combined = pd.read_csv("../combined.csv")

In [ ]:
combined.head().columns

Index(['Unnamed: 0', 'record_id', 'strand', 'source_file', 'search_term',
       'type', 'n_segments', 'qualifiers', 'exception', 'start1', 'end1',
       'start2', 'end2', 'slippage_site', 'slippage_direction', 'cluster',
       'accession', 'region_mol_type', 'region_strand', 'assembly_level',
       'assembly_name', 'submitter', 'total_sequence_length', 'gc_percent',
       'taxid', 'organism_name', 'release_date'],
      dtype='str')

In [ ]:
model_data = combined[['accession',
    'record_id',
    'cluster',
    'taxid',
    'slippage_site',
    'strand']]

In [ ]:
model_data.columns = ['accession', 'record_id', 'cluster', 'taxid', 'slippage_site', 'strand']

In [ ]:
model_data

,accession,record_id,cluster,taxid,slippage_site,strand
0,GCA_000852205.1,AB033998.1,AB033998.1,336960,3024,1
1,GCA_000898355.1,AB523788.1,AB523788.1,558690,6034,1
2,GCA_000891555.1,AB594828.1,NC_015050.1,909827,1620,1
3,GCA_000841865.2,AF022214.2,NC_001900.2,2905674,16074,1
4,GCA_000855425.1,AF033808.1,AF033808.1,11886,2481,1
...,...,...,...,...,...,...
124873,GCA_055197545.1,PZ005496.1,KF541239.1,11320,582,1
124874,GCA_055197675.1,PZ005504.1,KF541239.1,11320,582,1
124875,GCA_055198025.1,PZ005542.1,KF541239.1,11320,582,1
124876,GCA_003104755.1,U51190.1,AB253429.1,11676,1447,1


In [ ]:
import os

base_dir = '../viral_data_all/ncbi_dataset/data_subset'

def find_fna(accession):
    acc_dir = os.path.join(base_dir, accession)
    if os.path.isdir(acc_dir):
        for f in os.listdir(acc_dir):
            if f.endswith('_genomic.fna') and 'cds' not in f:
                return os.path.join(acc_dir, f)
    return None

model_data['fna_path'] = model_data['accession'].apply(find_fna)

# Check how many were found
print(f"Found: {model_data['fna_path'].notna().sum():,}")
print(f"Missing: {model_data['fna_path'].isna().sum():,}")

Found: 124,878
Missing: 0


In [ ]:
model_data

,accession,record_id,cluster,taxid,slippage_site,strand,fna_path
0,GCA_000852205.1,AB033998.1,AB033998.1,336960,3024,1,../viral_data_all/ncbi_dataset/data_subset/GCA...
1,GCA_000898355.1,AB523788.1,AB523788.1,558690,6034,1,../viral_data_all/ncbi_dataset/data_subset/GCA...
2,GCA_000891555.1,AB594828.1,NC_015050.1,909827,1620,1,../viral_data_all/ncbi_dataset/data_subset/GCA...
3,GCA_000841865.2,AF022214.2,NC_001900.2,2905674,16074,1,../viral_data_all/ncbi_dataset/data_subset/GCA...
4,GCA_000855425.1,AF033808.1,AF033808.1,11886,2481,1,../viral_data_all/ncbi_dataset/data_subset/GCA...
...,...,...,...,...,...,...,...
124873,GCA_055197545.1,PZ005496.1,KF541239.1,11320,582,1,../viral_data_all/ncbi_dataset/data_subset/GCA...
124874,GCA_055197675.1,PZ005504.1,KF541239.1,11320,582,1,../viral_data_all/ncbi_dataset/data_subset/GCA...
124875,GCA_055198025.1,PZ005542.1,KF541239.1,11320,582,1,../viral_data_all/ncbi_dataset/data_subset/GCA...
124876,GCA_003104755.1,U51190.1,AB253429.1,11676,1447,1,../viral_data_all/ncbi_dataset/data_subset/GCA...


In [ ]:
from sklearn.model_selection import train_test_split

train_idx, temp_idx = train_test_split(model_data.index, test_size=0.3, random_state=42)
val_idx, test_idx = train_test_split(temp_idx, test_size=0.667, random_state=42)

model_data['split'] = 'train'
model_data.loc[val_idx, 'split'] = 'val'
model_data.loc[test_idx, 'split'] = 'test'

print(model_data['split'].value_counts())

split
train    87414
test     24989
val      12475
Name: count, dtype: int64


In [ ]:
model_data.to_csv("../model_training.csv")

# MODEL TRAINING CSV

In [ ]:
model_data = pd.read_csv("../model_training.csv")

In [ ]:
model_data

,Unnamed: 0,accession,record_id,cluster,taxid,slippage_site,strand,fna_path
0,0,GCA_000852205.1,AB033998.1,AB033998.1,336960,3024,1,../viral_data_all/ncbi_dataset/data_subset/GCA...
1,1,GCA_000898355.1,AB523788.1,AB523788.1,558690,6034,1,../viral_data_all/ncbi_dataset/data_subset/GCA...
2,2,GCA_000891555.1,AB594828.1,NC_015050.1,909827,1620,1,../viral_data_all/ncbi_dataset/data_subset/GCA...
3,3,GCA_000841865.2,AF022214.2,NC_001900.2,2905674,16074,1,../viral_data_all/ncbi_dataset/data_subset/GCA...
4,4,GCA_000855425.1,AF033808.1,AF033808.1,11886,2481,1,../viral_data_all/ncbi_dataset/data_subset/GCA...
...,...,...,...,...,...,...,...,...
124873,124873,GCA_055197545.1,PZ005496.1,KF541239.1,11320,582,1,../viral_data_all/ncbi_dataset/data_subset/GCA...
124874,124874,GCA_055197675.1,PZ005504.1,KF541239.1,11320,582,1,../viral_data_all/ncbi_dataset/data_subset/GCA...
124875,124875,GCA_055198025.1,PZ005542.1,KF541239.1,11320,582,1,../viral_data_all/ncbi_dataset/data_subset/GCA...
124876,124876,GCA_003104755.1,U51190.1,AB253429.1,11676,1447,1,../viral_data_all/ncbi_dataset/data_subset/GCA...


In [ ]:
from ete3 import NCBITaxa

ncbi = NCBITaxa()

def get_taxonomic_lineage_info(taxid):
    """
    Given an NCBI TaxID, return two dictionaries:
      1) rank_to_taxid: { rank -> taxid }
      2) rank_to_name: { rank -> taxonomic name }
    """

    # Step 1: get lineage TaxIDs
    lineage_taxids = ncbi.get_lineage(taxid)

    # Step 2: get rank information
    taxid_to_rank = ncbi.get_rank(lineage_taxids)   # {taxid: rank}

    # Step 3: get names for all lineage taxids
    taxid_to_name = ncbi.get_taxid_translator(lineage_taxids)  # {taxid: name}

    # Step 4: build rank→taxid and rank→name mappings
    rank_to_taxid = {}
    rank_to_name = {}

    for tid in lineage_taxids:
        rank = taxid_to_rank.get(tid, None)
        if rank and rank != "no rank":
            rank_to_taxid[rank] = tid
            rank_to_name[rank] = taxid_to_name[tid]

    return rank_to_taxid, rank_to_name

def get_species_and_genus(taxid):
    """Extract species and genus names for a given taxid."""
    try:
        _, rank_to_name = get_taxonomic_lineage_info(taxid)
        species = rank_to_name.get("species", None)
        genus = rank_to_name.get("genus", None)
        return species, genus
    except Exception:
        return None, None

In [ ]:

# Apply to the dataframe
model_data[["species", "genus"]] = model_data["taxid"].apply(
    lambda x: pd.Series(get_species_and_genus(x))
)

In [ ]:
model_data

,Unnamed: 0,accession,record_id,cluster,taxid,slippage_site,strand,fna_path,species,genus
0,0,GCA_000852205.1,AB033998.1,AB033998.1,336960,3024,1,../viral_data_all/ncbi_dataset/data_subset/GCA...,Avastrovirus galli,Avastrovirus
1,1,GCA_000898355.1,AB523788.1,AB523788.1,558690,6034,1,../viral_data_all/ncbi_dataset/data_subset/GCA...,Cucurbit chlorotic yellows virus,Crinivirus
2,2,GCA_000891555.1,AB594828.1,NC_015050.1,909827,1620,1,../viral_data_all/ncbi_dataset/data_subset/GCA...,Polerovirus PEVYV1,Polerovirus
3,3,GCA_000841865.2,AF022214.2,NC_001900.2,2905674,16074,1,../viral_data_all/ncbi_dataset/data_subset/GCA...,Fromanvirus D29,Fromanvirus
4,4,GCA_000855425.1,AF033808.1,AF033808.1,11886,2481,1,../viral_data_all/ncbi_dataset/data_subset/GCA...,Alpharetrovirus avirousar,Alpharetrovirus
...,...,...,...,...,...,...,...,...,...,...
124873,124873,GCA_055197545.1,PZ005496.1,KF541239.1,11320,582,1,../viral_data_all/ncbi_dataset/data_subset/GCA...,Alphainfluenzavirus influenzae,Alphainfluenzavirus
124874,124874,GCA_055197675.1,PZ005504.1,KF541239.1,11320,582,1,../viral_data_all/ncbi_dataset/data_subset/GCA...,Alphainfluenzavirus influenzae,Alphainfluenzavirus
124875,124875,GCA_055198025.1,PZ005542.1,KF541239.1,11320,582,1,../viral_data_all/ncbi_dataset/data_subset/GCA...,Alphainfluenzavirus influenzae,Alphainfluenzavirus
124876,124876,GCA_003104755.1,U51190.1,AB253429.1,11676,1447,1,../viral_data_all/ncbi_dataset/data_subset/GCA...,Lentivirus humimdef1,Lentivirus


In [ ]:
model_data.to_csv("../model_training.csv")

# CREATING TRAINING DATASET 1

In [ ]:
cluster[cluster['cluster'] == cluster['ID']].value_counts()

cluster     ID        
MK072466.1  MK072466.1    2
MK072145.1  MK072145.1    2
MK072177.1  MK072177.1    2
MK072169.1  MK072169.1    2
MK072150.1  MK072150.1    2
                         ..
MH669001.1  MH669001.1    1
MZ375253.1  MZ375253.1    1
KP939613.1  KP939613.1    1
OP890528.1  OP890528.1    1
OQ469091.1  OQ469091.1    1
Name: count, Length: 50057, dtype: int64

In [ ]:
cluster['cluster'].unique()

<StringArray>
[ 'MW334171.1',  'KT030548.1',  'PX974797.1',  'OP414386.1',  'PP895394.1',
  'OQ504401.1', 'NC_036580.1',  'KT695110.1',  'LR031359.1',  'KX765863.1',
 ...
  'PX813953.1', 'NC_007903.1',  'MK871743.1',  'EU581825.1',  'KM101124.1',
  'MH669001.1',  'MZ375253.1',  'KP939613.1',  'OP890528.1',  'OQ469091.1']
Length: 50057, dtype: str

In [ ]:
model_data.iloc[0]

Unnamed: 0                                                       0
accession                                          GCA_000852205.1
record_id                                               AB033998.1
cluster                                                 AB033998.1
taxid                                                       336960
slippage_site                                                 3024
strand                                                           1
fna_path         ../viral_data_all/ncbi_dataset/data_subset/GCA...
species                                         Avastrovirus galli
genus                                                 Avastrovirus
Name: 0, dtype: object

In [ ]:
assembly

,accession,assembly_level,assembly_name,submitter,total_sequence_length,gc_percent,taxid,organism_name,release_date
0,GCA_000320725.1,Contig,APLentillevirus_1.0,CNRS UMR IRD 6236,1193433,28.0,1077221,Acanthamoeba polyphaga lentillevirus,2012-07-27
1,GCA_000847085.1,Complete Genome,ViralProj14573,"M. Jaeger, Dept of Virology, University of Ulm...",4491,33.5,1977403,Acholeplasma phage MV-L51,1993-04-21
2,GCA_000848265.1,Complete Genome,ViralProj14654,"NIH, NLM",5894,55.0,11788,Abelson murine leukemia virus,1998-01-22
3,GCA_000850225.1,Complete Genome,ViralProj14710,"BMC, University of Latvia",4268,44.0,154784,Acinetobacter phage AP205,2003-01-30
4,GCA_000858405.1,Complete Genome,ViralProj15222,NaN,5234,40.0,185639,Acheta domestica densovirus,2002-02-04
...,...,...,...,...,...,...,...,...,...
272510,GCF_000854525.1,Complete Genome,ViralProj14955,"Institut fuer Pflanzenvirologie, Mikrobiologie...",6624,53.5,253701,Zygocactus virus X,2004-07-21
272511,GCF_000861645.1,Complete Genome,ViralProj15390,"Department of Plant Pathology, National Chung ...",9591,43.0,12232,Zucchini yellow mosaic virus,2001-11-23
272512,GCF_004787635.1,Complete Genome,ASM478763v1,"Department of Microbiology & Immunobiology, Ha...",3160,37.0,114871,Zygosaccharomyces bailii virus Z,2023-05-04
272513,GCF_023124225.1,Complete Genome,ASM2312422v1,"Carrington Lab, Donald Danforth Plant Science ...",5969,51.5,2501222,Zymoseptoria tritici fusarivirus 1,2023-05-04


In [ ]:
accession_record = pd.read_csv("../accession_record_table.csv")

In [ ]:
accession_record

,accession,record_id
0,GCA_000320725.1,AFYC01000010.1
1,GCA_000320725.1,AFYC01000009.1
2,GCA_000320725.1,AFYC01000008.1
3,GCA_000320725.1,AFYC01000007.1
4,GCA_000320725.1,AFYC01000006.1
...,...,...
1443825,GCF_004787635.1,NC_075420.1
1443826,GCF_023124225.1,NC_076668.1
1443827,GCF_029888005.1,NC_078996.1
1443828,GCF_029888005.1,NC_078994.1


In [ ]:
cluster

,cluster,ID
0,MW334171.1,MW334171.1
1,MW334171.1,OP023516.1
2,MW334171.1,PQ755305.1
3,MW334171.1,PQ756818.1
4,MW334171.1,PQ756310.1
...,...,...
1326720,KP939613.1,KP939837.1
1326721,KP939613.1,KP940036.1
1326722,KP939613.1,KP940044.1
1326723,OP890528.1,OP890528.1


In [ ]:
cluster_temp = cluster[cluster['cluster'] == cluster['ID']].drop_duplicates()

In [ ]:
cluster_temp

,cluster,ID
0,MW334171.1,MW334171.1
522,KT030548.1,KT030548.1
692,PX974797.1,PX974797.1
693,OP414386.1,OP414386.1
694,PP895394.1,PP895394.1
...,...,...
1326681,MH669001.1,MH669001.1
1326683,MZ375253.1,MZ375253.1
1326685,KP939613.1,KP939613.1
1326723,OP890528.1,OP890528.1


In [ ]:
training_1 = cluster_temp.merge(accession_record, left_on = 'ID', right_on = 'record_id')

In [ ]:
training_1.value_counts()

cluster     ID          accession        record_id 
MW334171.1  MW334171.1  GCA_038804155.1  MW334171.1    1
KT030548.1  KT030548.1  GCA_003083835.1  KT030548.1    1
PX974797.1  PX974797.1  GCA_055130685.1  PX974797.1    1
OP414386.1  OP414386.1  GCA_025739875.1  OP414386.1    1
PP895394.1  PP895394.1  GCA_046887915.1  PP895394.1    1
                                                      ..
MH669001.1  MH669001.1  GCA_003442215.1  MH669001.1    1
MZ375253.1  MZ375253.1  GCA_020496375.1  MZ375253.1    1
KP939613.1  KP939613.1  GCA_003084335.1  KP939613.1    1
OP890528.1  OP890528.1  GCA_027132475.1  OP890528.1    1
OQ469091.1  OQ469091.1  GCA_029317805.1  OQ469091.1    1
Name: count, Length: 50180, dtype: int64

In [ ]:
duplicate_clusters = training_1[training_1.duplicated(subset=['cluster'], keep=False)]

In [ ]:
training_1 = training_1.drop_duplicates(subset=['cluster'], keep='first')

In [ ]:
training_1

,cluster,ID,accession,record_id
0,MW334171.1,MW334171.1,GCA_038804155.1,MW334171.1
1,KT030548.1,KT030548.1,GCA_003083835.1,KT030548.1
2,PX974797.1,PX974797.1,GCA_055130685.1,PX974797.1
3,OP414386.1,OP414386.1,GCA_025739875.1,OP414386.1
4,PP895394.1,PP895394.1,GCA_046887915.1,PP895394.1
...,...,...,...,...
50175,MH669001.1,MH669001.1,GCA_003442215.1,MH669001.1
50176,MZ375253.1,MZ375253.1,GCA_020496375.1,MZ375253.1
50177,KP939613.1,KP939613.1,GCA_003084335.1,KP939613.1
50178,OP890528.1,OP890528.1,GCA_027132475.1,OP890528.1


In [ ]:
training_1 = training_1.merge(assembly, how = 'left', on = 'accession')

In [ ]:
model_data

,Unnamed: 0,accession,record_id,cluster,taxid,slippage_site,strand,fna_path,species,genus
0,0,GCA_000852205.1,AB033998.1,AB033998.1,336960,3024,1,../viral_data_all/ncbi_dataset/data_subset/GCA...,Avastrovirus galli,Avastrovirus
1,1,GCA_000898355.1,AB523788.1,AB523788.1,558690,6034,1,../viral_data_all/ncbi_dataset/data_subset/GCA...,Cucurbit chlorotic yellows virus,Crinivirus
2,2,GCA_000891555.1,AB594828.1,NC_015050.1,909827,1620,1,../viral_data_all/ncbi_dataset/data_subset/GCA...,Polerovirus PEVYV1,Polerovirus
3,3,GCA_000841865.2,AF022214.2,NC_001900.2,2905674,16074,1,../viral_data_all/ncbi_dataset/data_subset/GCA...,Fromanvirus D29,Fromanvirus
4,4,GCA_000855425.1,AF033808.1,AF033808.1,11886,2481,1,../viral_data_all/ncbi_dataset/data_subset/GCA...,Alpharetrovirus avirousar,Alpharetrovirus
...,...,...,...,...,...,...,...,...,...,...
124873,124873,GCA_055197545.1,PZ005496.1,KF541239.1,11320,582,1,../viral_data_all/ncbi_dataset/data_subset/GCA...,Alphainfluenzavirus influenzae,Alphainfluenzavirus
124874,124874,GCA_055197675.1,PZ005504.1,KF541239.1,11320,582,1,../viral_data_all/ncbi_dataset/data_subset/GCA...,Alphainfluenzavirus influenzae,Alphainfluenzavirus
124875,124875,GCA_055198025.1,PZ005542.1,KF541239.1,11320,582,1,../viral_data_all/ncbi_dataset/data_subset/GCA...,Alphainfluenzavirus influenzae,Alphainfluenzavirus
124876,124876,GCA_003104755.1,U51190.1,AB253429.1,11676,1447,1,../viral_data_all/ncbi_dataset/data_subset/GCA...,Lentivirus humimdef1,Lentivirus


In [ ]:
training_1 = training_1.merge(model_data, how = 'left', on = 'record_id')

In [ ]:
training_1.columns

Index(['cluster_x', 'ID', 'accession_x', 'record_id', 'assembly_level',
       'assembly_name', 'submitter', 'total_sequence_length', 'gc_percent',
       'taxid_x', 'organism_name', 'release_date', 'Unnamed: 0', 'accession_y',
       'cluster_y', 'taxid_y', 'slippage_site', 'strand', 'fna_path',
       'species', 'genus'],
      dtype='str')

In [ ]:
training_1 = training_1[['accession_x', 'cluster_x', 'record_id', 'taxid_x', 'organism_name', 'slippage_site', 'strand', 'fna_path']]

In [ ]:
training_1_final = training_1

In [ ]:
training_1_final['strand'] = 1

In [ ]:
training_1_final.columns = ['accession', 'cluster', 'record_id', 'taxid', 'organism_name', 'slippage_site', 'strand', 'fna_path']

In [ ]:
training_1_final

,accession,cluster,record_id,taxid,organism_name,slippage_site,strand,fna_path
0,GCA_038804155.1,MW334171.1,MW334171.1,11320,Influenza A virus,NaN,1,NaN
1,GCA_003083835.1,KT030548.1,KT030548.1,86062,African horse sickness virus 8,NaN,1,NaN
2,GCA_055130685.1,PX974797.1,PX974797.1,11320,Influenza A virus,NaN,1,NaN
3,GCA_025739875.1,OP414386.1,OP414386.1,10244,Monkeypox virus,NaN,1,NaN
4,GCA_046887915.1,PP895394.1,PP895394.1,11320,Influenza A virus,570.0,1,../viral_data_all/ncbi_dataset/data_subset/GCA...
...,...,...,...,...,...,...,...,...
50052,GCA_003442215.1,MH669001.1,MH669001.1,2301563,Mycobacterium phage EleanorGeorge,9072.0,1,../viral_data_all/ncbi_dataset/data_subset/GCA...
50053,GCA_020496375.1,MZ375253.1,MZ375253.1,10760,Escherichia phage T7,NaN,1,NaN
50054,GCA_003084335.1,KP939613.1,KP939613.1,36421,African horse sickness virus 4,NaN,1,NaN
50055,GCA_027132475.1,OP890528.1,OP890528.1,10244,Monkeypox virus,NaN,1,NaN


In [ ]:
training_1_final['fna_path'][4]

'../viral_data_all/ncbi_dataset/data_subset/GCA_046887915.1/GCA_046887915.1_ASM4688791v1_genomic.fna'

In [ ]:
import os
import glob

def get_genomic_path(accession):
    # Construct the directory path
    base_dir = f"../viral_data_all/ncbi_dataset/data_subset/{accession}"
    
    # Use glob to find all files ending in .fna in that specific folder
    search_pattern = os.path.join(base_dir, "*.fna")
    files = glob.glob(search_pattern)
    
    # Filter the list: exclude files that contain 'cds' (case-insensitive)
    # We look at the basename to ensure we aren't accidentally filtering based on folder names
    genomic_files = [f for f in files if 'cds' not in os.path.basename(f).lower()]
    
    # Return the first match if it exists, otherwise return None or an empty string
    return genomic_files[0] if genomic_files else None

# Create the new column
training_1_final['fna_path'] = training_1_final['accession'].apply(get_genomic_path)

In [ ]:
training_1_final

,accession,cluster,record_id,taxid,organism_name,slippage_site,strand,fna_path
0,GCA_038804155.1,MW334171.1,MW334171.1,11320,Influenza A virus,NaN,1,../viral_data_all/ncbi_dataset/data_subset/GCA...
1,GCA_003083835.1,KT030548.1,KT030548.1,86062,African horse sickness virus 8,NaN,1,../viral_data_all/ncbi_dataset/data_subset/GCA...
2,GCA_055130685.1,PX974797.1,PX974797.1,11320,Influenza A virus,NaN,1,../viral_data_all/ncbi_dataset/data_subset/GCA...
3,GCA_025739875.1,OP414386.1,OP414386.1,10244,Monkeypox virus,NaN,1,../viral_data_all/ncbi_dataset/data_subset/GCA...
4,GCA_046887915.1,PP895394.1,PP895394.1,11320,Influenza A virus,570.0,1,../viral_data_all/ncbi_dataset/data_subset/GCA...
...,...,...,...,...,...,...,...,...
50052,GCA_003442215.1,MH669001.1,MH669001.1,2301563,Mycobacterium phage EleanorGeorge,9072.0,1,../viral_data_all/ncbi_dataset/data_subset/GCA...
50053,GCA_020496375.1,MZ375253.1,MZ375253.1,10760,Escherichia phage T7,NaN,1,../viral_data_all/ncbi_dataset/data_subset/GCA...
50054,GCA_003084335.1,KP939613.1,KP939613.1,36421,African horse sickness virus 4,NaN,1,../viral_data_all/ncbi_dataset/data_subset/GCA...
50055,GCA_027132475.1,OP890528.1,OP890528.1,10244,Monkeypox virus,NaN,1,../viral_data_all/ncbi_dataset/data_subset/GCA...


In [ ]:
training_1_final.to_csv("../training_1_final.csv")

# MAKING TEST FORM 1

In [ ]:
test_1 = cluster[cluster['cluster'] != cluster['ID']]

In [ ]:
test_1 = test_1.drop_duplicates(subset = ['cluster'], keep = 'first')

In [ ]:
accession_record

,accession,record_id
0,GCA_000320725.1,AFYC01000010.1
1,GCA_000320725.1,AFYC01000009.1
2,GCA_000320725.1,AFYC01000008.1
3,GCA_000320725.1,AFYC01000007.1
4,GCA_000320725.1,AFYC01000006.1
...,...,...
1443825,GCF_004787635.1,NC_075420.1
1443826,GCF_023124225.1,NC_076668.1
1443827,GCF_029888005.1,NC_078996.1
1443828,GCF_029888005.1,NC_078994.1


In [ ]:
test_1 = test_1.merge(accession_record, how = 'left', left_on = 'ID', right_on = 'record_id')

In [ ]:
test_1 = test_1[['cluster', 'record_id', 'accession']]

In [ ]:
assembly

,accession,assembly_level,assembly_name,submitter,total_sequence_length,gc_percent,taxid,organism_name,release_date
0,GCA_000320725.1,Contig,APLentillevirus_1.0,CNRS UMR IRD 6236,1193433,28.0,1077221,Acanthamoeba polyphaga lentillevirus,2012-07-27
1,GCA_000847085.1,Complete Genome,ViralProj14573,"M. Jaeger, Dept of Virology, University of Ulm...",4491,33.5,1977403,Acholeplasma phage MV-L51,1993-04-21
2,GCA_000848265.1,Complete Genome,ViralProj14654,"NIH, NLM",5894,55.0,11788,Abelson murine leukemia virus,1998-01-22
3,GCA_000850225.1,Complete Genome,ViralProj14710,"BMC, University of Latvia",4268,44.0,154784,Acinetobacter phage AP205,2003-01-30
4,GCA_000858405.1,Complete Genome,ViralProj15222,NaN,5234,40.0,185639,Acheta domestica densovirus,2002-02-04
...,...,...,...,...,...,...,...,...,...
272510,GCF_000854525.1,Complete Genome,ViralProj14955,"Institut fuer Pflanzenvirologie, Mikrobiologie...",6624,53.5,253701,Zygocactus virus X,2004-07-21
272511,GCF_000861645.1,Complete Genome,ViralProj15390,"Department of Plant Pathology, National Chung ...",9591,43.0,12232,Zucchini yellow mosaic virus,2001-11-23
272512,GCF_004787635.1,Complete Genome,ASM478763v1,"Department of Microbiology & Immunobiology, Ha...",3160,37.0,114871,Zygosaccharomyces bailii virus Z,2023-05-04
272513,GCF_023124225.1,Complete Genome,ASM2312422v1,"Carrington Lab, Donald Danforth Plant Science ...",5969,51.5,2501222,Zymoseptoria tritici fusarivirus 1,2023-05-04


In [ ]:
test_1 = test_1.merge(assembly, on = 'accession')

In [ ]:
test_1 = test_1[['cluster', 'record_id', 'accession', 'taxid', 'organism_name']]

In [ ]:
test_1['strand'] = 1

In [ ]:
test_1

,cluster,record_id,accession,taxid,organism_name,strand
0,MW334171.1,OP023516.1,GCA_039240725.1,11320,Influenza A virus,1
1,KT030548.1,KP939407.1,GCA_003081955.1,33714,African horse sickness virus 1,1
2,NC_036580.1,BK010347.1,GCA_002890315.1,2058778,Allium cepa amalgavirus 1,1
3,KT695110.1,KC580081.1,GCA_002650455.1,28875,Rotavirus A,1
4,KX765863.1,NC_052997.1,GCF_008894245.1,1897740,Salmonella phage ST-118,1
...,...,...,...,...,...,...
16431,MK871743.1,MK506983.1,GCA_039116645.1,11320,Influenza A virus,1
16432,KM101124.1,NC_026588.1,GCF_000954375.1,1527512,Mycobacterium phage Squirty,1
16433,MH669001.1,NC_051652.1,GCF_003442215.1,2301563,Mycobacterium phage EleanorGeorge,1
16434,MZ375253.1,MZ375251.1,GCA_020496355.1,10760,Escherichia phage T7,1


In [ ]:
import os
import glob

def get_genomic_path(accession):
    # Construct the directory path
    base_dir = f"../viral_data_all/ncbi_dataset/data_subset/{accession}"
    
    # Use glob to find all files ending in .fna in that specific folder
    search_pattern = os.path.join(base_dir, "*.fna")
    files = glob.glob(search_pattern)
    
    # Filter the list: exclude files that contain 'cds' (case-insensitive)
    # We look at the basename to ensure we aren't accidentally filtering based on folder names
    genomic_files = [f for f in files if 'cds' not in os.path.basename(f).lower()]
    
    # Return the first match if it exists, otherwise return None or an empty string
    return genomic_files[0] if genomic_files else None
    
test_1['fna_path'] = test_1['accession'].apply(get_genomic_path)

In [ ]:
test_1['slippage_site'] = None

In [ ]:
test_1['split'] = 'test'

In [ ]:
test_1 = test_1[['accession', 'cluster', 'record_id', 'taxid', 'split', 'slippage_site', 'strand', 'fna_path']]

In [ ]:
test_1

,accession,cluster,record_id,taxid,split,slippage_site,strand,fna_path
0,GCA_039240725.1,MW334171.1,OP023516.1,11320,test,None,1,../viral_data_all/ncbi_dataset/data_subset/GCA...
1,GCA_003081955.1,KT030548.1,KP939407.1,33714,test,None,1,../viral_data_all/ncbi_dataset/data_subset/GCA...
2,GCA_002890315.1,NC_036580.1,BK010347.1,2058778,test,None,1,../viral_data_all/ncbi_dataset/data_subset/GCA...
3,GCA_002650455.1,KT695110.1,KC580081.1,28875,test,None,1,../viral_data_all/ncbi_dataset/data_subset/GCA...
4,GCF_008894245.1,KX765863.1,NC_052997.1,1897740,test,None,1,../viral_data_all/ncbi_dataset/data_subset/GCF...
...,...,...,...,...,...,...,...,...
16431,GCA_039116645.1,MK871743.1,MK506983.1,11320,test,None,1,../viral_data_all/ncbi_dataset/data_subset/GCA...
16432,GCF_000954375.1,KM101124.1,NC_026588.1,1527512,test,None,1,../viral_data_all/ncbi_dataset/data_subset/GCF...
16433,GCF_003442215.1,MH669001.1,NC_051652.1,2301563,test,None,1,../viral_data_all/ncbi_dataset/data_subset/GCF...
16434,GCA_020496355.1,MZ375253.1,MZ375251.1,10760,test,None,1,../viral_data_all/ncbi_dataset/data_subset/GCA...


In [ ]:
training_1_final['split'] = 'train'

In [ ]:
training_1_final = training_1_final[['accession', 'cluster', 'record_id', 'taxid', 'split', 'slippage_site', 'strand', 'fna_path']]

In [ ]:
training_1_final 

,accession,cluster,record_id,taxid,split,slippage_site,strand,fna_path
0,GCA_038804155.1,MW334171.1,MW334171.1,11320,train,NaN,1,../viral_data_all/ncbi_dataset/data_subset/GCA...
1,GCA_003083835.1,KT030548.1,KT030548.1,86062,train,NaN,1,../viral_data_all/ncbi_dataset/data_subset/GCA...
2,GCA_055130685.1,PX974797.1,PX974797.1,11320,train,NaN,1,../viral_data_all/ncbi_dataset/data_subset/GCA...
3,GCA_025739875.1,OP414386.1,OP414386.1,10244,train,NaN,1,../viral_data_all/ncbi_dataset/data_subset/GCA...
4,GCA_046887915.1,PP895394.1,PP895394.1,11320,train,570.0,1,../viral_data_all/ncbi_dataset/data_subset/GCA...
...,...,...,...,...,...,...,...,...
50052,GCA_003442215.1,MH669001.1,MH669001.1,2301563,train,9072.0,1,../viral_data_all/ncbi_dataset/data_subset/GCA...
50053,GCA_020496375.1,MZ375253.1,MZ375253.1,10760,train,NaN,1,../viral_data_all/ncbi_dataset/data_subset/GCA...
50054,GCA_003084335.1,KP939613.1,KP939613.1,36421,train,NaN,1,../viral_data_all/ncbi_dataset/data_subset/GCA...
50055,GCA_027132475.1,OP890528.1,OP890528.1,10244,train,NaN,1,../viral_data_all/ncbi_dataset/data_subset/GCA...


In [ ]:
model_data_v1 = pd.concat([training_1_final, test_1], ignore_index=True)

In [ ]:
model_data_v1.iloc[0]['fna_path']

'../viral_data_all/ncbi_dataset/data_subset/GCA_038804155.1/GCA_038804155.1_ASM3880415v1_genomic.fna'

In [ ]:
model_data_v1.to_csv('../model_data_v1.csv')

In [ ]:
has_na = model_data_v1[model_data_v1['fna_path'].isna()]

In [ ]:
has_na

,accession,cluster,record_id,taxid,split,slippage_site,strand,fna_path


In [ ]:
has_na['fna_path'] = has_na['accession'].apply(get_genomic_path)

In [ ]:
has_na

,accession,cluster,record_id,taxid,split,slippage_site,strand,fna_path
53858,GCA_050518215.1,NC_038300.1,L36930.1,3052470,test,None,1,None
53995,GCA_050518215.1,NC_038298.1,L36929.1,3052470,test,None,1,None
64021,GCA_000844265.1,NC_000942.1,M22245.1,10530,test,None,1,None


In [ ]:
model_data_v1.at[53858, 'fna_path'] = '../viral_data_all/ncbi_dataset/data_subset/GCA_050518215.1/GCA_050518215.1_ASM5051821v1_genomic.fna'
model_data_v1.at[53995, 'fna_path'] = '../viral_data_all/ncbi_dataset/data_subset/GCA_050518215.1/GCA_050518215.1_ASM5051821v1_genomic.fna'

In [ ]:
model_data_v1.at[64021,'fna_path'] = '../viral_data_all/ncbi_dataset/data_subset/GCA_000844265.1/GCA_000844265.1_ViralProj14519_genomic.fna'

In [ ]:
model_data_v1.to_csv('../model_data_v1.csv')

In [ ]:
model_data_v1 = pd.read_csv("../model_data_v1.csv")

In [ ]:
model_data_v1

,Unnamed: 0,accession,cluster,record_id,taxid,split,slippage_site,strand,fna_path
0,0,GCA_038804155.1,MW334171.1,MW334171.1,11320,train,NaN,1,../viral_data_all/ncbi_dataset/data_subset/GCA...
1,1,GCA_003083835.1,KT030548.1,KT030548.1,86062,train,NaN,1,../viral_data_all/ncbi_dataset/data_subset/GCA...
2,2,GCA_055130685.1,PX974797.1,PX974797.1,11320,train,NaN,1,../viral_data_all/ncbi_dataset/data_subset/GCA...
3,3,GCA_025739875.1,OP414386.1,OP414386.1,10244,train,NaN,1,../viral_data_all/ncbi_dataset/data_subset/GCA...
4,4,GCA_046887915.1,PP895394.1,PP895394.1,11320,train,570.0,1,../viral_data_all/ncbi_dataset/data_subset/GCA...
...,...,...,...,...,...,...,...,...,...
66488,66488,GCA_039116645.1,MK871743.1,MK506983.1,11320,test,NaN,1,../viral_data_all/ncbi_dataset/data_subset/GCA...
66489,66489,GCF_000954375.1,KM101124.1,NC_026588.1,1527512,test,NaN,1,../viral_data_all/ncbi_dataset/data_subset/GCF...
66490,66490,GCF_003442215.1,MH669001.1,NC_051652.1,2301563,test,NaN,1,../viral_data_all/ncbi_dataset/data_subset/GCF...
66491,66491,GCA_020496355.1,MZ375253.1,MZ375251.1,10760,test,NaN,1,../viral_data_all/ncbi_dataset/data_subset/GCA...


In [ ]:
from ete3 import NCBITaxa

ncbi = NCBITaxa()

def get_taxonomic_lineage_info(taxid):
    """
    Given an NCBI TaxID, return two dictionaries:
      1) rank_to_taxid: { rank -> taxid }
      2) rank_to_name: { rank -> taxonomic name }
    """

    # Step 1: get lineage TaxIDs
    lineage_taxids = ncbi.get_lineage(taxid)

    # Step 2: get rank information
    taxid_to_rank = ncbi.get_rank(lineage_taxids)   # {taxid: rank}

    # Step 3: get names for all lineage taxids
    taxid_to_name = ncbi.get_taxid_translator(lineage_taxids)  # {taxid: name}

    # Step 4: build rank→taxid and rank→name mappings
    rank_to_taxid = {}
    rank_to_name = {}

    for tid in lineage_taxids:
        rank = taxid_to_rank.get(tid, None)
        if rank and rank != "no rank":
            rank_to_taxid[rank] = tid
            rank_to_name[rank] = taxid_to_name[tid]

    return rank_to_taxid, rank_to_name

def get_species_and_genus(taxid):
    """Extract species and genus names and taxids for a given taxid."""
    try:
        rank_to_taxid, rank_to_name = get_taxonomic_lineage_info(taxid)
        species = rank_to_name.get("species", None)
        species_id = rank_to_taxid.get("species", None)
        genus = rank_to_name.get("genus", None)
        genus_id = rank_to_taxid.get("genus", None)
        return species, species_id, genus, genus_id
    except Exception:
        return None, None, None, None


In [ ]:

# Apply to the dataframe
model_data_v1[["species", "species_id", "genus", "genus_id"]] = model_data_v1["taxid"].apply(
    lambda x: pd.Series(get_species_and_genus(x))
)



In [ ]:
model_data_v1

,Unnamed: 0,accession,cluster,record_id,taxid,split,slippage_site,strand,fna_path,species,species_id,genus,genus_id
0,0,GCA_038804155.1,MW334171.1,MW334171.1,11320,train,NaN,1,../viral_data_all/ncbi_dataset/data_subset/GCA...,Alphainfluenzavirus influenzae,2955291,Alphainfluenzavirus,197911.0
1,1,GCA_003083835.1,KT030548.1,KT030548.1,86062,train,NaN,1,../viral_data_all/ncbi_dataset/data_subset/GCA...,Orbivirus alphaequi,40050,Orbivirus,10892.0
2,2,GCA_055130685.1,PX974797.1,PX974797.1,11320,train,NaN,1,../viral_data_all/ncbi_dataset/data_subset/GCA...,Alphainfluenzavirus influenzae,2955291,Alphainfluenzavirus,197911.0
3,3,GCA_025739875.1,OP414386.1,OP414386.1,10244,train,NaN,1,../viral_data_all/ncbi_dataset/data_subset/GCA...,Orthopoxvirus monkeypox,3431483,Orthopoxvirus,10242.0
4,4,GCA_046887915.1,PP895394.1,PP895394.1,11320,train,570.0,1,../viral_data_all/ncbi_dataset/data_subset/GCA...,Alphainfluenzavirus influenzae,2955291,Alphainfluenzavirus,197911.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
66488,66488,GCA_039116645.1,MK871743.1,MK506983.1,11320,test,NaN,1,../viral_data_all/ncbi_dataset/data_subset/GCA...,Alphainfluenzavirus influenzae,2955291,Alphainfluenzavirus,197911.0
66489,66489,GCF_000954375.1,KM101124.1,NC_026588.1,1527512,test,NaN,1,../viral_data_all/ncbi_dataset/data_subset/GCF...,Squirtyvirus squirty,3432580,Squirtyvirus,2843461.0
66490,66490,GCF_003442215.1,MH669001.1,NC_051652.1,2301563,test,NaN,1,../viral_data_all/ncbi_dataset/data_subset/GCF...,Cheoctovirus eleanorgeorge,2955631,Cheoctovirus,1623281.0
66491,66491,GCA_020496355.1,MZ375253.1,MZ375251.1,10760,test,NaN,1,../viral_data_all/ncbi_dataset/data_subset/GCA...,Teseptimavirus T7,1985738,Teseptimavirus,110456.0


In [ ]:
model_data_v1.to_csv('../model_data_v1.csv')

In [ ]:
model_data_v1_final = pd.read_csv("../model_data_v1_string.csv")

In [ ]:
model_data_v1_final

,Unnamed: 0.1,Unnamed: 0,accession,cluster,record_id,taxid,split,slippage_site,strand,fna_path,species,species_id,genus,genus_id,sequence
0,0,0,GCA_038804155.1,MW334171.1,MW334171.1,11320,train,NaN,1,../viral_data_all/ncbi_dataset/data_subset/GCA...,Alphainfluenzavirus influenzae,2955291,Alphainfluenzavirus,197911.0,AGCAAAAGCAGGTCAATTATATTCAATATGGAGAGAATAAAAGAAC...
1,1,1,GCA_003083835.1,KT030548.1,KT030548.1,86062,train,NaN,1,../viral_data_all/ncbi_dataset/data_subset/GCA...,Orbivirus alphaequi,40050,Orbivirus,10892.0,GTTTAAAAATCCGTTCGTCATCATGGCAGAGGTCAGAAAGCAACAA...
2,2,2,GCA_055130685.1,PX974797.1,PX974797.1,11320,train,NaN,1,../viral_data_all/ncbi_dataset/data_subset/GCA...,Alphainfluenzavirus influenzae,2955291,Alphainfluenzavirus,197911.0,ATGGATGTCAATCCGACTCTATTGTTCCTAAAAGTTCCAGCGCAAA...
3,3,3,GCA_025739875.1,OP414386.1,OP414386.1,10244,train,NaN,1,../viral_data_all/ncbi_dataset/data_subset/GCA...,Orthopoxvirus monkeypox,3431483,Orthopoxvirus,10242.0,TNNTACAGATCATTTATATTCCAAAAATATTAACTATATACGTTTA...
4,4,4,GCA_046887915.1,PP895394.1,PP895394.1,11320,train,570.0,1,../viral_data_all/ncbi_dataset/data_subset/GCA...,Alphainfluenzavirus influenzae,2955291,Alphainfluenzavirus,197911.0,ATGGAAGACTTTGTGCGACAATGCTTCAATCCAATGGTCGTCGAGC...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
66488,66488,66488,GCA_039116645.1,MK871743.1,MK506983.1,11320,test,NaN,1,../viral_data_all/ncbi_dataset/data_subset/GCA...,Alphainfluenzavirus influenzae,2955291,Alphainfluenzavirus,197911.0,ATGGAGGCAGTAATAGTAGTCTTGCTATATACATTTGCAACTGTAG...
66489,66489,66489,GCF_000954375.1,KM101124.1,NC_026588.1,1527512,test,NaN,1,../viral_data_all/ncbi_dataset/data_subset/GCF...,Squirtyvirus squirty,3432580,Squirtyvirus,2843461.0,AGGCGAGTGTGTGTGTGACCCAGGGGTAGGGAGGTTGTAGTGCCGT...
66490,66490,66490,GCF_003442215.1,MH669001.1,NC_051652.1,2301563,test,NaN,1,../viral_data_all/ncbi_dataset/data_subset/GCF...,Cheoctovirus eleanorgeorge,2955631,Cheoctovirus,1623281.0,TGCAGATTTTGGTCTGTACGGAACCGGGGGGTTTCGCGGCATCCCC...
66491,66491,66491,GCA_020496355.1,MZ375253.1,MZ375251.1,10760,test,NaN,1,../viral_data_all/ncbi_dataset/data_subset/GCA...,Teseptimavirus T7,1985738,Teseptimavirus,110456.0,TCTCACAGTGTACGGACCTAAAGTTCCCCCATAGGGGGTACCTAAA...


accession_id,record_id,cluster_id,prf_position,strand,sequence,split,species_taxid,genus_taxid,species_name,genus_name

In [ ]:
model_data_v1_final = model_data_v1_final[['accession', 'record_id', 'cluster', 
'slippage_site', 'strand', 'sequence', 'split', 'species_id', 'genus_id', 'species', 'genus']]

In [ ]:
model_data_v1_final.columns = ['accession_id','record_id','cluster_id',
'prf_position','strand','sequence','split','species_taxid','genus_taxid','species_name','genus_name']

In [ ]:
model_data_v1_final

,accession_id,record_id,cluster_id,prf_position,strand,sequence,split,species_taxid,genus_taxid,species_name,genus_name
0,GCA_038804155.1,MW334171.1,MW334171.1,NaN,1,AGCAAAAGCAGGTCAATTATATTCAATATGGAGAGAATAAAAGAAC...,train,2955291,197911.0,Alphainfluenzavirus influenzae,Alphainfluenzavirus
1,GCA_003083835.1,KT030548.1,KT030548.1,NaN,1,GTTTAAAAATCCGTTCGTCATCATGGCAGAGGTCAGAAAGCAACAA...,train,40050,10892.0,Orbivirus alphaequi,Orbivirus
2,GCA_055130685.1,PX974797.1,PX974797.1,NaN,1,ATGGATGTCAATCCGACTCTATTGTTCCTAAAAGTTCCAGCGCAAA...,train,2955291,197911.0,Alphainfluenzavirus influenzae,Alphainfluenzavirus
3,GCA_025739875.1,OP414386.1,OP414386.1,NaN,1,TNNTACAGATCATTTATATTCCAAAAATATTAACTATATACGTTTA...,train,3431483,10242.0,Orthopoxvirus monkeypox,Orthopoxvirus
4,GCA_046887915.1,PP895394.1,PP895394.1,570.0,1,ATGGAAGACTTTGTGCGACAATGCTTCAATCCAATGGTCGTCGAGC...,train,2955291,197911.0,Alphainfluenzavirus influenzae,Alphainfluenzavirus
...,...,...,...,...,...,...,...,...,...,...,...
66488,GCA_039116645.1,MK506983.1,MK871743.1,NaN,1,ATGGAGGCAGTAATAGTAGTCTTGCTATATACATTTGCAACTGTAG...,test,2955291,197911.0,Alphainfluenzavirus influenzae,Alphainfluenzavirus
66489,GCF_000954375.1,NC_026588.1,KM101124.1,NaN,1,AGGCGAGTGTGTGTGTGACCCAGGGGTAGGGAGGTTGTAGTGCCGT...,test,3432580,2843461.0,Squirtyvirus squirty,Squirtyvirus
66490,GCF_003442215.1,NC_051652.1,MH669001.1,NaN,1,TGCAGATTTTGGTCTGTACGGAACCGGGGGGTTTCGCGGCATCCCC...,test,2955631,1623281.0,Cheoctovirus eleanorgeorge,Cheoctovirus
66491,GCA_020496355.1,MZ375251.1,MZ375253.1,NaN,1,TCTCACAGTGTACGGACCTAAAGTTCCCCCATAGGGGGTACCTAAA...,test,1985738,110456.0,Teseptimavirus T7,Teseptimavirus


In [ ]:
np.random.seed(42)
# Get test rows
test_mask = model_data_v1_final['split'] == 'test'
test_indices = model_data_v1_final[test_mask].index

# Randomly shuffle and split in half
shuffled = np.random.permutation(test_indices)
half = len(shuffled) // 2

# Assign half to 'val'
model_data_v1_final.loc[shuffled[:half], 'split'] = 'val'
# The other half remains 'test'

In [ ]:
model_data_v1_final['split'].value_counts()

split
train    50057
val       8218
test      8218
Name: count, dtype: int64

In [ ]:
has_na = model_data_v1_final[model_data_v1_final['sequence'].isna()]

In [ ]:
has_na

,Unnamed: 0.1,Unnamed: 0,accession,cluster,record_id,taxid,split,slippage_site,strand,fna_path,species,species_id,genus,genus_id,sequence
53858,53858,53858,GCA_050518215.1,NC_038300.1,L36930.1,3052470,test,NaN,1,../viral_data_all/ncbi_dataset/data_subset/GCA...,Orthohantavirus bayoui,3052470,Orthohantavirus,1980442.0,NaN
53995,53995,53995,GCA_050518215.1,NC_038298.1,L36929.1,3052470,test,NaN,1,../viral_data_all/ncbi_dataset/data_subset/GCA...,Orthohantavirus bayoui,3052470,Orthohantavirus,1980442.0,NaN
64021,64021,64021,GCA_000844265.1,NC_000942.1,M22245.1,10530,test,NaN,1,../viral_data_all/ncbi_dataset/data_subset/GCA...,Mastadenovirus encephalomyelitidis,3241422,Mastadenovirus,10509.0,NaN


In [ ]:
lengths = model_data_v1_final['sequence'].str.len().values

In [ ]:
length_comparison = pd.DataFrame({
    'lengths': lengths,
    'prf_position': model_data_v1_final['prf_position']
})

In [ ]:
temp = length_comparison.dropna()

In [ ]:
(temp.iloc[:, 0] > temp.iloc[:, 1]).all()

np.True_

In [ ]:
model_data_v1_final

,accession_id,record_id,cluster_id,prf_position,strand,sequence,split,species_taxid,genus_taxid,species_name,genus_name
0,GCA_038804155.1,MW334171.1,MW334171.1,NaN,1,AGCAAAAGCAGGTCAATTATATTCAATATGGAGAGAATAAAAGAAC...,train,2955291,197911.0,Alphainfluenzavirus influenzae,Alphainfluenzavirus
1,GCA_003083835.1,KT030548.1,KT030548.1,NaN,1,GTTTAAAAATCCGTTCGTCATCATGGCAGAGGTCAGAAAGCAACAA...,train,40050,10892.0,Orbivirus alphaequi,Orbivirus
2,GCA_055130685.1,PX974797.1,PX974797.1,NaN,1,ATGGATGTCAATCCGACTCTATTGTTCCTAAAAGTTCCAGCGCAAA...,train,2955291,197911.0,Alphainfluenzavirus influenzae,Alphainfluenzavirus
3,GCA_025739875.1,OP414386.1,OP414386.1,NaN,1,TNNTACAGATCATTTATATTCCAAAAATATTAACTATATACGTTTA...,train,3431483,10242.0,Orthopoxvirus monkeypox,Orthopoxvirus
4,GCA_046887915.1,PP895394.1,PP895394.1,570.0,1,ATGGAAGACTTTGTGCGACAATGCTTCAATCCAATGGTCGTCGAGC...,train,2955291,197911.0,Alphainfluenzavirus influenzae,Alphainfluenzavirus
...,...,...,...,...,...,...,...,...,...,...,...
66488,GCA_039116645.1,MK506983.1,MK871743.1,NaN,1,ATGGAGGCAGTAATAGTAGTCTTGCTATATACATTTGCAACTGTAG...,test,2955291,197911.0,Alphainfluenzavirus influenzae,Alphainfluenzavirus
66489,GCF_000954375.1,NC_026588.1,KM101124.1,NaN,1,AGGCGAGTGTGTGTGTGACCCAGGGGTAGGGAGGTTGTAGTGCCGT...,val,3432580,2843461.0,Squirtyvirus squirty,Squirtyvirus
66490,GCF_003442215.1,NC_051652.1,MH669001.1,NaN,1,TGCAGATTTTGGTCTGTACGGAACCGGGGGGTTTCGCGGCATCCCC...,test,2955631,1623281.0,Cheoctovirus eleanorgeorge,Cheoctovirus
66491,GCA_020496355.1,MZ375251.1,MZ375253.1,NaN,1,TCTCACAGTGTACGGACCTAAAGTTCCCCCATAGGGGGTACCTAAA...,val,1985738,110456.0,Teseptimavirus T7,Teseptimavirus


In [ ]:
df = model_data_v1_final.dropna(subset=['sequence'])

In [ ]:
df.iloc[4]['sequence']

'ATGGAAGACTTTGTGCGACAATGCTTCAATCCAATGGTCGTCGAGCTTGCGGAAAAGGCAATGAAAGAATATGGAGAAGATCCGAAAATTGAGACAAACAAATTTGCTGCAATATGCACACACATGGAAGTCTGTTTCATGTATTCTGATTTCCATTTCATCGATGAGAGAGGGGAACCAACAATAGTAGAATCCAGTGACCCAAATGCACTCCTAAAACACAGATTTGAAATAATTGAGGGAAGAGACCGCGCAATGGCTTGGACGGTGGTGAACAGCATTTGCAATACTACAGGGGTTGAGAAGCCCAAGTTCCTCCCAGATCTATATGACTATAAGGAAAACAGGTTCATTGAAATCGGTGTAACAAGAAGGGAAGTGCACATATACTACCTAGAGAAAGCTAACAAAATAAAATCAGAGAAAACACACATCCATATTTTCTCGTTCACTGGTGAAGAAATGGCTACCAAGGCAGATTACACTCTTGACGAGGAAAGCAGAGCAAGAATCAAGACTAGGTTATTCACAATCAGGCAGGAAATGGCCATCAGGGCCCTATGGGACTCCTTTCGTCAGTCCGAAAGGGGCGAGGAGACAGTTGAAGAAAGATTTGACATCAAGGGAACCATGCATAGGCTTGCCAACCAGAGTCTCCCGCCGAACTTCTCCAGTCTTGAAAACTTTAGAGCCTATGTGGATGGGTTCGAACCGAACGGCTGCATTGAGGGCAAGCTTTCTCAAATGTCCAAAGGAGTGAATGCCAGAATTGAGCCATTTTTGAGGGAAACACCACGCCCTCTTAGATTACCTGATGGACCTCCCTGTTTTCAGCGGTCAAAATTCCTGTTGATGGACGCTTTAAAACTGAGCATTGAAGATCCGAGTCATGAAGGAGAGGGAATACCACTATATGATGCAATCAAGTGCATGAAGACATTTTTCGGATGGAAAGAGCCCCACATTGTCAAGCCGCATGAGAAAGGCATAAATCCAAAC

In [ ]:
df['split'].value_counts()

split
train    50057
val       8217
test      8216
Name: count, dtype: int64

In [ ]:
df.to_csv('../model_data_v1_final.csv')